# FTS Model Complexity Sweep & Out-of-Sample Backtest Comparison

This notebook demonstrates how to systematically evaluate model architecture variations (e.g. `num_layers` in an LSTM) under **controlled isolation** (*ceteris paribus*).

### Core Principles:
1. **Fixed Baseline Hyperparameters:** All non-swept hyperparameters (learning rate, hidden dimension, batch size, dropout, execution fees, slippage) are held strictly constant.
2. **Out-of-Sample Evaluation:** Models are trained on the training split, registered as candidate ONNX models in `ModelRegistryLog`, and backtested on an independent holdout test split using the `BacktestEngine`.
3. **Complexity-vs-Performance Curve:** We pair validation statistical metrics (`Val IC`) with Out-of-Sample trading metrics (`OOS Sharpe`, `Max Drawdown`) to pinpoint model capacity sweet-spots and detect overfitting.

### 1. Import Dependencies & Set Pathing

In [ ]:
import os
import sys
import logging
import pandas as pd
import matplotlib.pyplot as plt

# Ensure src and project modules are on path
sys.path.insert(0, os.path.abspath("../src"))

from trading_bot.config import settings
from plugins.nets.training.sweep_runner import run_parameter_sweep

# Set logging level
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger()

### 2. Execute Controlled Parameter Sweep

We load the sweep specification from `specs/train/BTCUSDT/sweep/lstm_num_layers.yaml` and iterate through `num_layers = [1, 2, 3, 4]`.

In [ ]:
spec_path = "../specs/train/BTCUSDT/sweep/lstm_num_layers.yaml"
results_df = run_parameter_sweep(spec_path)

print("\n=================== OUT-OF-SAMPLE SWEEP RESULTS ===================")
display(results_df)

### 3. Complexity vs. Performance Sensitivity Analysis

We plot **Validation IC (Statistical Fit)** alongside **Out-of-Sample Sharpe Ratio (Strategy Performance)** across `num_layers` to inspect model capacity and identify the overfitting threshold.

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))

color = 'tab:blue'
ax1.set_xlabel('LSTM Num Layers (Complexity)', fontsize=12)
ax1.set_ylabel('Validation IC', color=color, fontsize=12)
ax1.plot(results_df['num_layers'], results_df['val_ic'], color=color, marker='o', linewidth=2, label='Val IC')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, linestyle='--', alpha=0.5)

ax2 = ax1.twinx()
color = 'tab:green'
ax2.set_ylabel('OOS Sharpe Ratio', color=color, fontsize=12)
ax2.plot(results_df['num_layers'], results_df['oos_sharpe'], color=color, marker='s', linewidth=2, linestyle='--', label='OOS Sharpe')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('LSTM Model Capacity vs. Strategy Performance', fontsize=14, pad=15)
fig.tight_layout()
plt.show()